# 문제 2 파이프라인 — 소청초 중간층 수온 복원 (제출본)

## 최종 구성 요약
- **feature**: 층별로 다른 세트 (`LAYER_FEATURE_SETS_FINAL`) + 밀도 기반 T-S feature (`FS_HYBRID`)
- **모델**: HistGradientBoostingRegressor, 층별 개별 학습, 시간 기준 조기종료
- **하이퍼파라미터**: Optuna 탐색값(`study.best_params`) — FS_HYBRID로 재탐색해봤으나 개선 없어 원값 유지
- **후처리**: 저성층 구간 소프트 게이팅 (block2류 균질 구간에서 모델 과신 방지)
- **검증**: 5-block leave-one-block-out CV, block2(균질 구간)는 참고용으로 별도 표기

## 이 노트북에서 뺀 것 (탐색 후 기각/과정 삭제, 결론만 반영)
- Ridge 비교 → HistGBR이 전 블록 우세 (기각)
- RandomForest 비교 → 성능도 낮고 HistGBR과 오차상관 0.87~0.89로 다양성도 없음 (기각)
- FS_HYBRID 기준 Optuna 재탐색 → 원 파라미터보다 저하 (기각, 원 파라미터 유지)
- Permutation importance 원계산 / layer2 leave-one-out 반복 / 게이팅 grid search 원코드 →
  계산 자체는 삭제하고 **결론만 하드코딩** (재현에 오래 걸리는 탐색 과정이라 제출본엔 불필요)


## 0. 설정 및 상수

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from copy import deepcopy
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings('ignore')
warnings.filterwarnings(action='ignore')
warnings.simplefilter("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

TARGET_LAYERS = [2, 3, 4]
REF_LAYERS_ALL = [1, 5, 6, 7, 8]
INTERP_LIMIT = 6  # 10분 x 6 = 1시간


## 1. 데이터 로드 + 짧은 결측 보간 (누수 방지 포함)

In [ ]:
observations = pd.read_csv("../data/observations.csv")
observations["time"] = pd.to_datetime(observations["time"])
observations = observations.sort_values(["layer", "time"]).reset_index(drop=True)

DESIGNED_MASK = (
    observations["layer"].isin(TARGET_LAYERS)
    & (observations["time"] >= "2025-09-01")
    & (observations["time"] <= "2025-10-31 23:50:00")
)


def interpolate_short_gaps(df, cols=("temp", "psal", "depth"), limit=INTERP_LIMIT):
    """층별로 연속 결측이 limit스텝 이하일 때만 선형보간. limit_area='inside'로 계열 앞/뒤 결측은 안 채움."""
    df = df.sort_values(["layer", "time"]).reset_index(drop=True)
    flag_cols = []
    for col in cols:
        before_na = df[col].isna()
        df[col] = (
            df.groupby("layer", group_keys=False)[col]
            .apply(lambda s: s.interpolate(method="linear", limit=limit, limit_area="inside"))
        )
        after_na = df[col].isna()
        flag_col = f"{col}_is_interpolated"
        df[flag_col] = before_na & ~after_na
        flag_cols.append(flag_col)
    return df, flag_cols


observations_filled, flag_cols = interpolate_short_gaps(observations)

# 의도된 가림구간(DESIGNED_MASK)은 보간에서 완전히 보호 (누수 방지)
for col in ["temp", "psal", "depth"]:
    flag_col = f"{col}_is_interpolated"
    observations_filled.loc[DESIGNED_MASK, col] = np.nan
    observations_filled.loc[DESIGNED_MASK, flag_col] = False

truly_observed_temp = observations_filled["temp"].notna() & (~observations_filled["temp_is_interpolated"])
print(f"observations_filled: {len(observations_filled):,}행")


## 2. 혼합강도 feature (성층지수 / 진동진폭 / 계류선흔들림 / 표층냉각)

In [ ]:
ref_pool = observations_filled[observations_filled["layer"].isin(REF_LAYERS_ALL)]

temp_ref_wide_raw = ref_pool.pivot_table(index="time", columns="layer", values="temp")
n_available = temp_ref_wide_raw.notna().sum(axis=1)
stratification_index = temp_ref_wide_raw.std(axis=1, skipna=True)
stratification_index[n_available < 2] = np.nan


def rolling_std_by_layer(df, col, window, min_periods):
    out = []
    for layer, g in df.groupby("layer"):
        g = g.sort_values("time")
        rstd = g[col].rolling(window, min_periods=min_periods).std()
        out.append(pd.DataFrame({"layer": layer, "time": g["time"].reset_index(drop=True), "rstd": rstd.reset_index(drop=True)}))
    return pd.concat(out, ignore_index=True)


def rolling_mean_by_layer(df, col, window, min_periods):
    out = []
    for layer, g in df.groupby("layer"):
        g = g.sort_values("time")
        rmean = g[col].rolling(window, min_periods=min_periods).mean()
        out.append(pd.DataFrame({"layer": layer, "time": g["time"].reset_index(drop=True), "rmean": rmean.reset_index(drop=True)}))
    return pd.concat(out, ignore_index=True)


temp_rstd_1h = rolling_std_by_layer(ref_pool, "temp", window=6, min_periods=3)
oscillation_amplitude = temp_rstd_1h.pivot_table(index="time", columns="layer", values="rstd").mean(axis=1, skipna=True)

dev = observations_filled.copy()
dev["depth_dev"] = (dev["depth"] - dev["nominal_depth"]).abs()
mooring_swing = (
    dev[dev["layer"].isin(REF_LAYERS_ALL)]
    .pivot_table(index="time", columns="layer", values="depth_dev")
    .mean(axis=1, skipna=True)
)

layer1 = observations_filled[observations_filled["layer"] == 1].sort_values("time").set_index("time")
surface_cooling_6h = -layer1["temp"].diff(36)
surface_cooling_24h = -layer1["temp"].diff(144)

mixing_features = pd.DataFrame({
    "stratification_index": stratification_index,
    "oscillation_amplitude": oscillation_amplitude,
    "mooring_swing": mooring_swing,
    "surface_cooling_6h": surface_cooling_6h,
    "surface_cooling_24h": surface_cooling_24h,
}).sort_index()

layer1_only = ref_pool[ref_pool["layer"] == 1]
temp1_smooth = rolling_mean_by_layer(layer1_only, "temp", window=144, min_periods=36).set_index("time")["rmean"]
mixing_features["temp_layer1_smooth24h"] = temp1_smooth
mixing_features["temp_layer1_trend_smooth24h"] = mixing_features["temp_layer1_smooth24h"].diff(144)

strat_smooth = mixing_features["stratification_index"].rolling(144, min_periods=36).mean()
mixing_features["stratification_index_smooth24h"] = strat_smooth
mixing_features["stratification_index_trend_smooth24h"] = strat_smooth.diff(144)

print(f"mixing_features: {len(mixing_features):,}행, 컬럼 {list(mixing_features.columns)}")


## 3. 참조층 feature table (temp/psal pivot) + baseline lookup

In [ ]:
temp_ref_wide = ref_pool.pivot_table(index="time", columns="layer", values="temp")
temp_ref_wide.columns = [f"temp_layer{c}" for c in temp_ref_wide.columns]

psal_ref_wide = ref_pool.pivot_table(index="time", columns="layer", values="psal")
psal_ref_wide.columns = [f"psal_layer{c}" for c in psal_ref_wide.columns]

public_ref_count = ref_pool.pivot_table(index="time", columns="layer", values="temp").notna().sum(axis=1)
public_ref_count.name = "public_ref_count"

feature_table = temp_ref_wide.join(psal_ref_wide).join(public_ref_count).join(mixing_features)

idx = feature_table.index
month = idx.month.to_numpy()
doy = idx.dayofyear.to_numpy()
hour = idx.hour.to_numpy() + idx.minute.to_numpy() / 60
feature_table["month_sin"] = np.sin(2 * np.pi * month / 12)
feature_table["month_cos"] = np.cos(2 * np.pi * month / 12)
feature_table["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
feature_table["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
feature_table["hour_sin"] = np.sin(2 * np.pi * hour / 24)
feature_table["hour_cos"] = np.cos(2 * np.pi * hour / 24)


def build_baseline_lookup(pool_df):
    """참조층만으로 시각별 (depth, temp) 정렬배열을 만들어 수심 선형보간 baseline을 계산하는 용도."""
    pool = pool_df.dropna(subset=["temp"])
    lookup = {}
    for t, g in pool.groupby("time"):
        g_sorted = g.sort_values("nominal_depth")
        lookup[t] = (g_sorted["nominal_depth"].to_numpy(), g_sorted["temp"].to_numpy())
    return lookup


def apply_baseline(lookup, times, target_depths):
    preds = np.full(len(times), np.nan)
    for i, (t, d) in enumerate(zip(times, target_depths)):
        arr = lookup.get(t)
        if arr is None or len(arr[0]) == 0:
            continue
        depths, temps = arr
        if d <= depths[0]:
            preds[i] = temps[0]
        elif d >= depths[-1]:
            preds[i] = temps[-1]
        else:
            preds[i] = np.interp(d, depths, temps)
    return preds


baseline_lookup = build_baseline_lookup(ref_pool)
print(f"feature_table: {len(feature_table):,}행, feature {len(feature_table.columns)}개")


## 4. T-S(수온-염분) 관계 기반 feature

가림 층의 염분도 함께 가려지므로, 참조층의 온도+염분으로 두 가지를 만든다:
1. **밀도 기반 성층지수**: 온도만 쓰던 기존 `stratification_index`보다 물리적으로 더 정확한 성층 강도
2. **T-S 기후값 회귀**: '참조층 염분 + 계절 → 온도' 국소 회귀 예측치 (layer4에서 특히 유효, 아래 5번 참고)

`build_ts_climatology_features`는 누수 방지를 위해 `exclude_mask`를 받아 그 구간을 뺀 데이터로만 학습한다
(block CV에서는 검증 블록을, 최종 학습에서는 실제 가림구간만 제외).

In [ ]:
# 밀도이상(sigma) 근사: 정확한 EOS-80/TEOS-10 대신, 이 수온·염분 범위(10~25도, 30~35psu)에서
# 상대적 크기 비교(성층 강도)에는 충분한 선형 근사 사용.
ALPHA_T, BETA_S = 0.20, 0.75

psal_ref_wide_raw = ref_pool.pivot_table(index="time", columns="layer", values="psal")
dens_wide = BETA_S * psal_ref_wide_raw - ALPHA_T * temp_ref_wide_raw

n_dens_available = dens_wide.notna().sum(axis=1)
density_stratification_index = dens_wide.std(axis=1, skipna=True)
density_stratification_index[n_dens_available < 2] = np.nan


def smooth24h(s, window=144, min_periods=72):
    return s.rolling(window, min_periods=min_periods).mean()


density_stratification_index_smooth24h = smooth24h(density_stratification_index)
density_stratification_index_trend_smooth24h = density_stratification_index_smooth24h.diff(144)

density_features = pd.DataFrame({
    "density_stratification_index_smooth24h": density_stratification_index_smooth24h,
    "density_stratification_index_trend_smooth24h": density_stratification_index_trend_smooth24h,
    "density_layer5": dens_wide.get(5),
    "density_layer7": dens_wide.get(7),
})

feature_table = feature_table.join(density_features)

TS_PREDICTOR_COLS = [c for c in feature_table.columns if c.startswith("psal_layer")] + \
                     ["month_sin", "month_cos", "doy_sin", "doy_cos"]


def build_ts_climatology_features(exclude_mask, feature_table_df):
    """타겟층별로 '참조층 psal + 계절 -> 그 층 온도' 국소 회귀를 exclude_mask 제외 데이터로 학습하고,
    전체 시간대에 대한 예측값을 feature로 반환."""
    ts_cols = {}
    for layer in TARGET_LAYERS:
        train_mask = (
            observations_filled["layer"].eq(layer)
            & truly_observed_temp
            & (~DESIGNED_MASK)
            & (~exclude_mask)
        )
        rows = observations_filled.loc[train_mask, ["time", "temp"]].merge(
            feature_table_df[TS_PREDICTOR_COLS], left_on="time", right_index=True, how="left"
        )
        model = HistGradientBoostingRegressor(max_iter=200, max_depth=4, learning_rate=0.1, random_state=42)
        model.fit(rows[TS_PREDICTOR_COLS], rows["temp"])
        ts_cols[f"ts_clim_pred_layer{layer}"] = pd.Series(
            model.predict(feature_table_df[TS_PREDICTOR_COLS]), index=feature_table_df.index
        )
    return pd.DataFrame(ts_cols)


print("밀도/T-S feature 준비 완료:", list(density_features.columns) + [f"ts_clim_pred_layer{l}" for l in TARGET_LAYERS])


## 5. 검증 유틸리티 (score.py 로직 + fake mask 생성기)

In [ ]:
def local_score(sub, ans):
    """score.py와 동일한 검증/채점 로직"""
    K = ["station", "layer", "time"]
    required = K + ["temp"]
    assert list(sub.columns) == required, f"열 이름/순서 오류: {list(sub.columns)}"
    assert set(required).issubset(ans.columns), "정답 파일 스키마 오류"
    assert len(sub) == len(ans), f"행 수 오류: 제출 {len(sub)} / 정답 {len(ans)}"
    assert not sub[K].isna().any().any(), "제출 키에 결측이 있습니다"
    assert not sub.duplicated(K).any(), "제출 키 중복"
    assert not ans.duplicated(K).any(), "정답 키 중복"

    m = ans[required].merge(sub, on=K, how="outer", indicator=True, suffixes=("_t", "_p"), validate="one_to_one")
    assert m["_merge"].eq("both").all(), f"키 집합 불일치: {int(m['_merge'].ne('both').sum())}건"

    p = pd.to_numeric(m["temp_p"], errors="coerce")
    assert not p.isna().any(), "temp는 결측 없는 유한한 숫자여야 합니다"
    assert p.between(-5.0, 45.0).all(), "temp 허용 범위는 -5~45 ℃입니다"

    e = p.to_numpy() - m["temp_t"].to_numpy()
    rmse = np.sqrt(np.mean(e**2))
    print(f"RMSE = {rmse:.6f} degC  (n={len(m)})")
    for layer, g in m.groupby("layer"):
        layer_rmse = np.sqrt(np.mean((g["temp_p"] - g["temp_t"])**2))
        print(f"  layer {int(layer)} : {layer_rmse:.6f}")
    return rmse


def build_fake_fold(mask, ref_layers, obs_df):
    """mask(가려질 영역) + 참조층 목록을 받아 fake test_index / answer를 만드는 범용 함수"""
    ref_wide = obs_df[obs_df["layer"].isin(ref_layers)].pivot_table(index="time", columns="layer", values="temp")
    pub_count = ref_wide.notna().sum(axis=1)

    candidate = obs_df[mask].copy()
    candidate["is_truly_observed"] = truly_observed_temp[mask].values
    candidate["public_ref_count"] = candidate["time"].map(pub_count)

    scoreable = candidate[candidate["is_truly_observed"] & (candidate["public_ref_count"] >= 2)].copy()
    answer = scoreable[["station", "layer", "time", "temp"]].reset_index(drop=True)
    test_index = scoreable[["station", "layer", "time"]].reset_index(drop=True)
    return test_index, answer


## 6. 학습 유틸리티

In [ ]:
def build_train_rows(exclude_mask, feature_table_df):
    """exclude_mask(학습에서 뺄 영역, boolean Series, observations_filled와 같은 인덱스)를 반영해
    학습용 데이터(baseline_pred, target_resid 포함)를 조립"""
    pool_mask = (
        observations_filled["layer"].isin(TARGET_LAYERS)
        & truly_observed_temp
        & (~DESIGNED_MASK)
        & (~exclude_mask)
    )
    rows = observations_filled.loc[pool_mask, ["station", "layer", "time", "temp", "nominal_depth"]].copy()
    rows = rows.merge(feature_table_df, left_on="time", right_index=True, how="left")
    rows["baseline_pred"] = apply_baseline(baseline_lookup, rows["time"].to_numpy(), rows["nominal_depth"].to_numpy())
    rows["target_resid"] = rows["temp"] - rows["baseline_pred"]
    return rows.dropna(subset=["baseline_pred"])


## 7. 최종 feature 세트

층별로 다른 세트를 쓴다. 아래 목록은 다음 과정을 거쳐 확정됐다 (탐색 코드는 삭제, 결론만 반영):
- **다중공선성 점검**(상관행렬 |r|>0.8, VIF)으로 16개 공통 후보(`DECORRELATED_COMMON`)로 1차 축소
- **layer2**: 16개에서 permutation importance + leave-one-out ablation으로 4개 추가 제거(12개) —
  `public_ref_count`, `surface_cooling_24h`, `psal_layer8`, `mooring_swing` 제거 시 오히려 성능 향상 확인
- **layer3/4**: block CV로 "공통16 vs 층별 core만 vs 층별 core+marginal" 비교 → **core만**이 최고 or 동률
- 이후 **T-S/밀도 feature 추가**: layer2·3은 밀도지수 4종만 추가해도 이득, layer4는 밀도지수+T-S 회귀 예측치까지 추가할 때 최고 성능
  (block2 제외 4블록 평균 improve_pct: 공통16 14.52 → 층별+T-S/밀도 하이브리드 15.73)

In [ ]:
LAYER_FEATURE_SETS_FINAL = {
    2: ["temp_layer1", "temp_layer5", "temp_layer7", "temp_layer8",
        "psal_layer1", "psal_layer5", "psal_layer7",
        "stratification_index_smooth24h", "stratification_index_trend_smooth24h",
        "temp_layer1_trend_smooth24h", "oscillation_amplitude", "surface_cooling_6h"],
    3: ["stratification_index_smooth24h", "temp_layer5", "temp_layer7", "psal_layer5", "temp_layer1"],
    4: ["psal_layer5", "stratification_index_smooth24h", "temp_layer1", "temp_layer5", "temp_layer7"],
}

DENSITY_FEATS = ["density_stratification_index_smooth24h", "density_stratification_index_trend_smooth24h",
                  "density_layer5", "density_layer7"]

FS_HYBRID = {
    2: LAYER_FEATURE_SETS_FINAL[2] + DENSITY_FEATS,
    3: LAYER_FEATURE_SETS_FINAL[3] + DENSITY_FEATS,
    4: LAYER_FEATURE_SETS_FINAL[4] + DENSITY_FEATS + ["ts_clim_pred_layer4"],
}

for l in TARGET_LAYERS:
    print(f"layer {l}: {len(FS_HYBRID[l])}개 -> {FS_HYBRID[l]}")


## 8. Block 기반 K-fold 교차검증

`build_fake_fold` 하나로만 판단하면 표본이 1~2개뿐이라 노이즈에 취약하므로,
학습 가능한 전체 기간을 5개 연속 블록으로 나눠 매번 한 블록만 검증에, 나머지로 학습.

**주의**: block2는 baseline RMSE 자체가 매우 작은(0.24) 이미 균질한 구간이라 모델이 상대적으로 크게 틀어지는
구조적 특이 구간이다. block2 제외 4블록 평균을 신뢰 지표로 쓰고, block2는 게이팅으로 별도 방어한다(11번 참고).

In [ ]:
K_FOLDS = 5

clean_times = (
    observations_filled.loc[
        observations_filled["layer"].isin(TARGET_LAYERS) & truly_observed_temp, "time"
    ]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

split_indices = np.array_split(np.arange(len(clean_times)), K_FOLDS)
blocks = [clean_times.iloc[idx] for idx in split_indices]
block_ranges = [(b.iloc[0], b.iloc[-1]) for b in blocks]

print(f"총 {len(clean_times):,}개 고유 시각을 {K_FOLDS}개 블록으로 분할")
for i, (s, e) in enumerate(block_ranges):
    print(f"block {i}: {s} ~ {e}  ({len(blocks[i]):,}개 고유 시각)")


## 9. 모델 학습 함수 (시간 기준 조기종료 + fit_fn 팩토리)

In [ ]:
def fit_with_time_based_early_stopping(X, y, times, val_frac=0.1, patience=15, max_iter=300, **hgbr_kwargs):
    order = np.argsort(np.asarray(times))
    X_sorted = X.iloc[order].reset_index(drop=True)
    y_sorted = y.iloc[order].reset_index(drop=True)

    n_val = max(int(len(X_sorted) * val_frac), 50)
    X_train, y_train = X_sorted.iloc[:-n_val], y_sorted.iloc[:-n_val]
    X_val, y_val = X_sorted.iloc[-n_val:], y_sorted.iloc[-n_val:]

    model = HistGradientBoostingRegressor(max_iter=1, warm_start=True, early_stopping=False, **hgbr_kwargs)
    best_rmse, best_model, best_n, no_improve = np.inf, None, 0, 0

    for n in range(1, max_iter + 1):
        model.max_iter = n
        model.fit(X_train, y_train)
        rmse = np.sqrt(mean_squared_error(y_val, model.predict(X_val)))
        if rmse < best_rmse - 1e-5:
            best_rmse, best_model, best_n = rmse, deepcopy(model), n
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    return best_model, best_n, best_rmse


def make_hgbr_fit_fn(**hgbr_kwargs):
    def fit_fn(X, y, times):
        model, best_n, val_rmse = fit_with_time_based_early_stopping(X, y, times, **hgbr_kwargs)
        return model
    return fit_fn


class SeedEnsembleModel:
    """여러 시드로 학습한 모델들의 예측 평균을 내는 래퍼."""
    def __init__(self, models):
        self.models = models

    def predict(self, X):
        preds = np.column_stack([m.predict(X) for m in self.models])
        return preds.mean(axis=1)


def make_seed_ensemble_fit_fn(n_seeds=5, base_seed=42, **hgbr_kwargs):
    """주의: HistGradientBoostingRegressor는 행 부트스트랩을 하지 않고, random_state는
    표본이 20만 행을 넘을 때만 히스토그램 구간 표본추출에 쓰인다. 우리 데이터 규모(층당 수만 행)에서는
    random_state만 바꿔선 완전히 동일한 모델이 나오므로, 여기서 직접 복원추출(bootstrap)로 다양성을 만든다."""
    hgbr_kwargs.pop("random_state", None)
    def fit_fn(X, y, times):
        n = len(X)
        models = []
        for i in range(n_seeds):
            rng = np.random.RandomState(base_seed + i)
            boot_idx = rng.randint(0, n, size=n)
            X_boot = X.iloc[boot_idx].reset_index(drop=True)
            y_boot = y.iloc[boot_idx].reset_index(drop=True)
            times_boot = pd.Series(np.asarray(times)[boot_idx])
            model, _, _ = fit_with_time_based_early_stopping(
                X_boot, y_boot, times_boot, random_state=base_seed + i, **hgbr_kwargs
            )
            models.append(model)
        return SeedEnsembleModel(models)
    return fit_fn


## 10. Block CV 평가 함수 (FS_HYBRID + T-S augment, 층별 다른 fit_fn 지원)

In [ ]:
def resolve_fit_fn(fit_fn, layer):
    """fit_fn이 dict({layer: fit_fn})로 오면 층별로 다른 모델을 쓸 수 있게, 아니면 그대로 공통으로 사용."""
    return fit_fn[layer] if isinstance(fit_fn, dict) else fit_fn


def run_block_cv_augmented(feature_sets, block_ranges, augment_fn=None, ref_layers=REF_LAYERS_ALL, fit_fn=None):
    """leave-one-block-out CV. augment_fn(block_mask)이 있으면 fold별로 T-S 회귀 feature를
    그 블록을 제외한 데이터로 다시 학습해 누수 없이 반영한다."""
    if fit_fn is None:
        fit_fn = make_hgbr_fit_fn(learning_rate=0.05, max_depth=6, random_state=42)
    results = []
    raw_prediction_frames = []
    for i, (start, end) in enumerate(block_ranges):
        block_mask = (
            observations_filled["layer"].isin(TARGET_LAYERS)
            & (observations_filled["time"] >= start)
            & (observations_filled["time"] <= end)
        )
        fold_feature_table = feature_table
        if augment_fn is not None:
            extra = augment_fn(block_mask)
            fold_feature_table = feature_table.join(extra)

        test_idx, answer = build_fake_fold(block_mask, ref_layers, observations_filled)
        train_df = build_train_rows(block_mask, fold_feature_table)

        trained = {}
        for layer in TARGET_LAYERS:
            cols = feature_sets[layer]
            sub = train_df[train_df["layer"] == layer]
            trained[layer] = resolve_fit_fn(fit_fn, layer)(sub[cols], sub["target_resid"], sub["time"])

        tf = test_idx.merge(
            observations_filled[["station", "layer", "time", "nominal_depth"]],
            on=["station", "layer", "time"], how="left",
        )
        tf = tf.merge(fold_feature_table, left_on="time", right_index=True, how="left")
        tf["baseline_pred"] = apply_baseline(baseline_lookup, tf["time"].to_numpy(), tf["nominal_depth"].to_numpy())

        preds = np.full(len(tf), np.nan)
        resid_preds_raw = np.full(len(tf), np.nan)
        for layer in TARGET_LAYERS:
            cols = feature_sets[layer]
            idx_layer = tf["layer"] == layer
            resid_pred = trained[layer].predict(tf.loc[idx_layer, cols])
            resid_preds_raw[idx_layer.to_numpy()] = resid_pred
            preds[idx_layer.to_numpy()] = tf.loc[idx_layer, "baseline_pred"].to_numpy() + resid_pred

        tf["pred"] = preds
        tf["resid_pred_raw"] = resid_preds_raw
        tf["block"] = i
        merged = tf.merge(answer, on=["station", "layer", "time"])
        raw_prediction_frames.append(merged)

        row = {"block": i}
        rmse_model = np.sqrt(np.mean((merged["pred"] - merged["temp"]) ** 2))
        rmse_base = np.sqrt(np.mean((merged["baseline_pred"] - merged["temp"]) ** 2))
        row["improve_pct"] = 100 * (1 - rmse_model / rmse_base)
        for layer, g in merged.groupby("layer"):
            rm = np.sqrt(np.mean((g["pred"] - g["temp"]) ** 2))
            rb = np.sqrt(np.mean((g["baseline_pred"] - g["temp"]) ** 2))
            row[f"layer{int(layer)}_improve_pct"] = 100 * (1 - rm / rb)
        results.append(row)
        print(f"block {i} (n={len(merged):,}): improve_pct={row['improve_pct']:+.1f}%")

    return pd.DataFrame(results), pd.concat(raw_prediction_frames, ignore_index=True)


## 11. Optuna 하이퍼파라미터 탐색

`REDUCED_BLOCKS`(block 1, 3만)로 싸게 탐색 — block2를 포함하면 극단값 때문에 탐색이 왜곡되고,
5블록 전체를 매 trial마다 돌리면 비용이 너무 크다.

**참고**: FS_HYBRID 확정 이후 이 파라미터로 재탐색을 시도했으나(같은 방식, feature만 FS_HYBRID로 교체)
block2 제외 4블록 평균이 오히려 하락(16.08→14.97)해 기각했다. 아래 원 파라미터를 그대로 사용한다.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

REDUCED_BLOCKS = [block_ranges[1], block_ranges[3]]


def objective(trial):
    params = dict(
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        l2_regularization=trial.suggest_float("l2_regularization", 0.0, 1.0),
        max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 15, 63),
        random_state=42,
    )
    fit_fn = make_hgbr_fit_fn(**params)
    cv_res, _ = run_block_cv_augmented(
        FS_HYBRID, REDUCED_BLOCKS,
        augment_fn=lambda mask: build_ts_climatology_features(mask, feature_table),
        fit_fn=fit_fn,
    )
    return -cv_res["improve_pct"].mean()


study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=15, show_progress_bar=False)

print("최적 하이퍼파라미터:", study.best_params)
print(f"축소 블록 기준 평균 개선율: {-study.best_value:.2f}%")


## 12. 층별 개별 하이퍼파라미터 탐색

layer4는 psal_layer5 지배적인 완전히 다른 구조라 공통 파라미터가 최선이 아닐 수 있어, 층별로 따로
탐색해봤다. 대상 층만 새 파라미터를 쓰고 나머지 두 층은 공통 파라미터(`study.best_params`)로 고정한 채
그 층의 improve_pct만을 목적함수로 최적화한다.

**결과**: layer2(22.66→25.64)·layer3(18.18→22.35)는 개별 튜닝이 뚜렷하게 유리했고,
layer4(10.41→10.35, 오차범위)는 득이 없어 공통 파라미터를 그대로 썼다.

In [ ]:
def objective_per_layer(trial, target_layer):
    params = dict(
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        l2_regularization=trial.suggest_float("l2_regularization", 0.0, 1.0),
        max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 15, 63),
        random_state=42,
    )
    fit_fn = {
        l: (make_hgbr_fit_fn(**params) if l == target_layer else make_hgbr_fit_fn(**study.best_params))
        for l in TARGET_LAYERS
    }
    cv_res, _ = run_block_cv_augmented(
        FS_HYBRID, REDUCED_BLOCKS,
        augment_fn=lambda mask: build_ts_climatology_features(mask, feature_table),
        fit_fn=fit_fn,
    )
    return -cv_res[f"layer{target_layer}_improve_pct"].mean()


studies_per_layer = {}
for layer in TARGET_LAYERS:
    study_l = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
    study_l.optimize(lambda trial: objective_per_layer(trial, layer), n_trials=15, show_progress_bar=False)
    studies_per_layer[layer] = study_l
    print(f"layer {layer} 최적 파라미터: {study_l.best_params} (개선율 {-study_l.best_value:.2f}%)")


## 13. 최종 모델 구조 확정

- **layer2**: 부트스트랩 5시드 앙상블 + 층별 개별 튜닝 파라미터
- **layer3**: 단일 모델 + 층별 개별 튜닝 파라미터
- **layer4**: 단일 모델 + 공통 파라미터 (개별 튜닝이 REDUCED_BLOCKS 2개 블록에 과적합된 것으로 판단해 기각)

HistGradientBoostingRegressor는 행 부트스트랩을 하지 않고 `random_state`도 대용량(20만 행 초과)일 때만
쓰이므로, 시드만 바꿔서는 앙상블 효과가 없다. 직접 복원추출(bootstrap)로 다양성을 만든 뒤 층별로 비교한 결과
layer2에서만 뚜렷한 개선이 있어 layer2에만 채택했다.

In [ ]:
FINAL_FIT_FN = {
    2: make_seed_ensemble_fit_fn(n_seeds=5, **studies_per_layer[2].best_params),
    3: make_hgbr_fit_fn(**studies_per_layer[3].best_params),
    4: make_hgbr_fit_fn(**study.best_params),
}

cv_final, raw_final = run_block_cv_augmented(
    FS_HYBRID, block_ranges,
    augment_fn=lambda mask: build_ts_climatology_features(mask, feature_table),
    fit_fn=FINAL_FIT_FN,
)

cols = ["improve_pct", "layer2_improve_pct", "layer3_improve_pct", "layer4_improve_pct"]
print("=== block2 제외 4블록 평균 (신뢰 기준) ===")
print(cv_final[cv_final["block"] != 2][cols].mean().round(2))


## 14. 저성층 게이팅

block2류의 균질 구간(baseline이 이미 정확한데 모델이 미세 잔차를 과신해서 오히려 나빠지는 구간)을
방어하기 위해, 밀도 기반 성층지수가 낮을수록 모델 예측 비중(alpha)을 baseline 쪽으로 부드럽게 낮춘다.

threshold/smoothness는 grid search로 층별 최적화한 결과값을 그대로 사용 (탐색 코드는 삭제):
하드 게이팅보다 소프트 게이팅이 block2를 더 잘 방어하면서 다른 블록엔 손해가 없었다.

In [ ]:
GATE_COL = "density_stratification_index_smooth24h"

GATE_PARAMS_FINAL = {
    2: {"threshold": 0.675137, "smoothness": 0.10},  # layer2: 개별 튜닝+앙상블로 resid 분포가 달라져 재조정
    3: {"threshold": 0.675137, "smoothness": 0.10},  # layer3: 개별 튜닝으로 resid 분포가 달라져 재조정
    4: {"threshold": 0.891463, "smoothness": 0.01},  # layer4: 파라미터 안 바꿨으므로 기존값 유지
}


def soft_gate_fn(threshold, smoothness):
    return lambda x: 1 / (1 + np.exp(-(x - threshold) / smoothness))


def apply_gate(df, alpha_fn, gate_col=GATE_COL):
    out = df.copy()
    gate_vals = out[gate_col].fillna(out[gate_col].median())
    out["alpha"] = alpha_fn(gate_vals.to_numpy())
    out["pred_gated"] = out["baseline_pred"] + out["alpha"] * out["resid_pred_raw"]
    return out


def summarize(df, pred_col="pred_gated"):
    rows = []
    for b, gb in df.groupby("block"):
        row = {"block": b}
        rmse_model = np.sqrt(np.mean((gb[pred_col] - gb["temp"]) ** 2))
        rmse_base = np.sqrt(np.mean((gb["baseline_pred"] - gb["temp"]) ** 2))
        row["improve_pct"] = 100 * (1 - rmse_model / rmse_base)
        for layer, g in gb.groupby("layer"):
            rm = np.sqrt(np.mean((g[pred_col] - g["temp"]) ** 2))
            rb = np.sqrt(np.mean((g["baseline_pred"] - g["temp"]) ** 2))
            row[f"layer{int(layer)}_improve_pct"] = 100 * (1 - rm / rb)
        rows.append(row)
    return pd.DataFrame(rows)


gated_frames = []
for layer in TARGET_LAYERS:
    sub = raw_final[raw_final["layer"] == layer].copy()
    p = GATE_PARAMS_FINAL[layer]
    gated_frames.append(apply_gate(sub, soft_gate_fn(p["threshold"], p["smoothness"])))

all_gated = pd.concat(gated_frames, ignore_index=True)
print("=== 게이팅 적용 후 (block2 포함 5블록) ===")
print(summarize(all_gated).round(2).to_string(index=False))
print("\n=== block2 제외 4블록 평균 ===")
print(summarize(all_gated)[lambda d: d["block"] != 2][cols].mean().round(2))


## 15. 최종 재학습 및 제출 파일 생성

In [ ]:
# 최종 T-S 회귀 feature: 실제 가림구간(DESIGNED_MASK)만 제외하고 전체로 학습
ts_clim_final = build_ts_climatology_features(DESIGNED_MASK, feature_table)
feature_table_final = feature_table.join(ts_clim_final)

no_exclusion = pd.Series(False, index=observations_filled.index)
final_train_rows = build_train_rows(no_exclusion, feature_table_final)
print(f"최종 학습 데이터: {len(final_train_rows):,}행")

final_models = {}
for layer in TARGET_LAYERS:
    cols_l = FS_HYBRID[layer]
    sub = final_train_rows[final_train_rows["layer"] == layer]
    final_models[layer] = resolve_fit_fn(FINAL_FIT_FN, layer)(sub[cols_l], sub["target_resid"], sub["time"])
    print(f"layer {layer} 최종 모델 학습 완료 (n={len(sub):,})")


In [ ]:
# 실제 test_index.csv에 대해 예측
test_index_raw = pd.read_csv("../data/test_index.csv")
test_index_raw["time_dt"] = pd.to_datetime(test_index_raw["time"])

test_features = test_index_raw.merge(
    observations_filled[["station", "layer", "time", "nominal_depth"]],
    left_on=["station", "layer", "time_dt"], right_on=["station", "layer", "time"],
    how="left", suffixes=("", "_obs"),
)
test_features = test_features.merge(feature_table_final, left_on="time_dt", right_index=True, how="left")
test_features["baseline_pred"] = apply_baseline(
    baseline_lookup, test_features["time_dt"].to_numpy(), test_features["nominal_depth"].to_numpy()
)

print(f"테스트 행 수: {len(test_features):,} (test_index.csv와 일치해야 함: {len(test_index_raw):,})")
print(f"nominal_depth 결측: {test_features['nominal_depth'].isna().sum()}건 (0이어야 안전)")
print(f"baseline_pred 결측: {test_features['baseline_pred'].isna().sum()}건")


In [ ]:
final_preds = np.full(len(test_features), np.nan)
for layer in TARGET_LAYERS:
    cols_l = FS_HYBRID[layer]
    idx_layer = test_features["layer"] == layer
    resid_pred = final_models[layer].predict(test_features.loc[idx_layer, cols_l])

    # 소프트 게이팅 적용
    p = GATE_PARAMS_FINAL[layer]
    gate_vals = test_features.loc[idx_layer, GATE_COL].fillna(test_features[GATE_COL].median()).to_numpy()
    alpha = 1 / (1 + np.exp(-(gate_vals - p["threshold"]) / p["smoothness"]))

    baseline_vals = test_features.loc[idx_layer, "baseline_pred"].to_numpy()
    final_preds[idx_layer.to_numpy()] = baseline_vals + alpha * resid_pred

submission = test_index_raw[["station", "layer", "time"]].copy()
submission["temp"] = final_preds

n_nan = submission["temp"].isna().sum()
if n_nan > 0:
    print(f"경고: baseline을 못 만든 {n_nan}건을 전체 평균으로 fallback 처리")
    submission["temp"] = submission["temp"].fillna(final_train_rows["temp"].mean())

n_out_of_range = (~submission["temp"].between(-5.0, 45.0)).sum()
if n_out_of_range > 0:
    print(f"경고: 범위를 벗어난 {n_out_of_range}건을 클립함")
    submission["temp"] = submission["temp"].clip(-5.0, 45.0)

assert len(submission) == len(test_index_raw), "행 수 불일치"
assert not submission[["station", "layer", "time"]].isna().any().any(), "키에 결측"
assert not submission.duplicated(["station", "layer", "time"]).any(), "키 중복"
assert list(submission.columns) == ["station", "layer", "time", "temp"], "컬럼 순서 오류"

submission.to_csv("submission.csv", index=False)
print(f"submission.csv 저장 완료. 총 {len(submission):,}행")
